In [51]:
!pip install tensorflow
!pip install catboost

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from catboost import CatBoostRegressor
import joblib

In [52]:
df = pd.read_csv('ipl_data.csv')
print(df.shape)
print(df.head())


(76014, 15)
   mid        date                  venue               bat_team  \
0    1  2008-04-18  M Chinnaswamy Stadium  Kolkata Knight Riders   
1    1  2008-04-18  M Chinnaswamy Stadium  Kolkata Knight Riders   
2    1  2008-04-18  M Chinnaswamy Stadium  Kolkata Knight Riders   
3    1  2008-04-18  M Chinnaswamy Stadium  Kolkata Knight Riders   
4    1  2008-04-18  M Chinnaswamy Stadium  Kolkata Knight Riders   

                     bowl_team      batsman   bowler  runs  wickets  overs  \
0  Royal Challengers Bangalore   SC Ganguly  P Kumar     1        0    0.1   
1  Royal Challengers Bangalore  BB McCullum  P Kumar     1        0    0.2   
2  Royal Challengers Bangalore  BB McCullum  P Kumar     2        0    0.2   
3  Royal Challengers Bangalore  BB McCullum  P Kumar     2        0    0.3   
4  Royal Challengers Bangalore  BB McCullum  P Kumar     2        0    0.4   

   runs_last_5  wickets_last_5  striker  non-striker  total  
0            1               0        0         

In [53]:
# Use only data after 5 overs
df = df[df['overs'] >= 5.0]

print("Dataset Shape:", df.shape)

Dataset Shape: (56707, 15)


In [54]:

X = df[
    [
        'venue',
        'bat_team',
        'bowl_team',
        'runs',
        'wickets',
        'overs',
        'runs_last_5',
        'wickets_last_5'
    ]
]

y = df['total']

In [55]:

print(X.dtypes)

print("\nMissing Values:")
print(X.isnull().sum())

venue              object
bat_team           object
bowl_team          object
runs                int64
wickets             int64
overs             float64
runs_last_5         int64
wickets_last_5      int64
dtype: object

Missing Values:
venue             0
bat_team          0
bowl_team         0
runs              0
wickets           0
overs             0
runs_last_5       0
wickets_last_5    0
dtype: int64


In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)


Training Samples : (45365, 8)
Testing Samples  : (11342, 8)


In [57]:
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_state=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=['venue', 'bat_team', 'bowl_team']
)


0:	learn: 28.3652505	total: 169ms	remaining: 2m 49s
100:	learn: 15.1743400	total: 3.87s	remaining: 34.5s
200:	learn: 13.3059967	total: 7.34s	remaining: 29.2s
300:	learn: 12.2622688	total: 11s	remaining: 25.5s
400:	learn: 11.4721331	total: 14.8s	remaining: 22.2s
500:	learn: 10.9218747	total: 18.7s	remaining: 18.6s
600:	learn: 10.4527704	total: 22.5s	remaining: 14.9s
700:	learn: 10.0796038	total: 26.2s	remaining: 11.2s
800:	learn: 9.7498721	total: 29.9s	remaining: 7.44s
900:	learn: 9.4694753	total: 33.6s	remaining: 3.69s
999:	learn: 9.1902797	total: 37.1s	remaining: 0us


CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.05, loss_function='RMSE', random_state=42, verbose=100)

In [58]:
preds = model.predict(X_test)

In [59]:
mae = mean_absolute_error(y_test, preds)

rmse = np.sqrt(
    mean_squared_error(y_test, preds)
)

r2 = r2_score(y_test, preds)

print("\nModel Performance")
print("-----------------------")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R²   :", r2)



Model Performance
-----------------------
MAE  : 6.683866331213941
RMSE : 9.595371080615429
R²   : 0.8925878745662759


In [60]:
joblib.dump(model, "ipl_score_predictor.pkl")

print("\nModel saved successfully!")


Model saved successfully!


In [ ]:
teams = sorted(df['bat_team'].unique())
venues = sorted(df['venue'].unique())

joblib.dump(teams, "teams.pkl")
joblib.dump(venues, "venues.pkl")

In [63]:
sample = pd.DataFrame({
    'venue': ['M Chinnaswamy Stadium'],
    'bat_team': ['Royal Challengers Bangalore'],
    'bowl_team': ['Chennai Super Kings'],
    'runs': [80],
    'wickets': [2],
    'overs': [10.0],
    'runs_last_5': [45],
    'wickets_last_5': [1]
})

predicted_score = model.predict(sample)

print("Predicted Final Score:", round(predicted_score[0]))

Predicted Final Score: 178
